# Australian Bitcoin Purchase Cost Analyzer


In [ ]:
# Install dependencies
!pip install -q requests pandas matplotlib numpy


# Australian Bitcoin Purchase Cost Analyzer

This notebook helps Australian investors compare the total cost of buying Bitcoin across different exchanges and payment methods.

Based on analysis of major Australian exchanges and their fee structures, we calculate the real net cost including maker/taker fees, payment method surcharges, and spreads.

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

try:
    response = requests.get('https://api.coingecko.com/api/v3/simple/price?ids=bitcoin&vs_currencies=aud')
    btc_aud = response.json()['bitcoin']['aud']
except:
    btc_aud = 75000

print(f'Current BTC/AUD price: ${btc_aud:,.2f} (as of {datetime.now().strftime("%Y-%m-%d %H:%M")})')

## Australian Exchange Fee Structures

Based on typical 2026 fee models for major Australian Bitcoin exchanges:

In [ ]:
exchanges = {
    'Exchange A': {'maker_fee': 0.002, 'taker_fee': 0.004, 'bpay_fee': 0.01, 'cc_fee': 0.025},
    'Exchange B': {'maker_fee': 0.001, 'taker_fee': 0.0025, 'bpay_fee': 0.005, 'cc_fee': 0.03},
    'Exchange C': {'maker_fee': 0.0025, 'taker_fee': 0.005, 'bpay_fee': 0.02, 'cc_fee': 0.02},
}

payment_methods = ['BPAY', 'Credit Card', 'Bank Transfer']
method_fee_map = {'BPAY': 'bpay_fee', 'Credit Card': 'cc_fee', 'Bank Transfer': 'taker_fee'}

print('Exchange Fee Structures (as percentage):')
for ex, fees in exchanges.items():
    print(f'\n{ex}:')
    print(f'  Maker: {fees["maker_fee"]*100:.2f}% | Taker: {fees["taker_fee"]*100:.2f}%')
    print(f'  BPAY surcharge: {fees["bpay_fee"]*100:.2f}% | Credit Card: {fees["cc_fee"]*100:.2f}%')

In [ ]:
btc_amounts = [0.1, 0.5, 1.0]
results = []

for btc_qty in btc_amounts:
    aud_cost = btc_qty * btc_aud
    
    for ex_name, ex_fees in exchanges.items():
        for payment in payment_methods:
            if payment == 'Bank Transfer':
                fee_pct = ex_fees[method_fee_map[payment]]
            elif payment == 'BPAY':
                fee_pct = ex_fees[method_fee_map[payment]]
            else:
                fee_pct = ex_fees[method_fee_map[payment]]
            
            total_cost = aud_cost * (1 + fee_pct)
            fee_amount = total_cost - aud_cost
            cost_per_btc = total_cost / btc_qty
            
            results.append({
                'BTC Quantity': btc_qty,
                'Exchange': ex_name,
                'Payment Method': payment,
                'Base Cost (AUD)': aud_cost,
                'Total Fees (AUD)': fee_amount,
                'Total Cost (AUD)': total_cost,
                'Cost per BTC (AUD)': cost_per_btc,
                'Fee %': fee_pct * 100
            })

df = pd.DataFrame(results)
print(f'\nCost Analysis for {btc_amounts[1]} BTC purchase:\n')
print(df[df['BTC Quantity'] == 0.5][['Exchange', 'Payment Method', 'Total Cost (AUD)', 'Fee %']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_05 = df[df['BTC Quantity'] == 0.5]
pivot_cost = df_05.pivot_table(values='Total Cost (AUD)', index='Exchange', columns='Payment Method')
pivot_cost.plot(kind='bar', ax=axes[0], color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[0].set_title('Total Cost by Exchange & Payment Method (0.5 BTC)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Cost (AUD)')
axes[0].set_xlabel('Exchange')
axes[0].legend(title='Payment Method', fontsize=9)
axes[0].grid(axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

df_fee = df[df['BTC Quantity'] == 0.5]
pivot_fee = df_fee.pivot_table(values='Fee %', index='Exchange', columns='Payment Method')
pivot_fee.plot(kind='bar', ax=axes[1], color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[1].set_title('Fee Percentage by Exchange & Payment Method', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Fee (%)')
axes[1].set_xlabel('Exchange')
axes[1].legend(title='Payment Method', fontsize=9)
axes[1].grid(axis='y', alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('\nChart generated: Fee comparison across exchanges and payment methods.')

## Summary: Cheapest and Most Expensive Options

In [ ]:
summary = df[df['BTC Quantity'] == 0.5].copy()
summary['Option'] = summary['Exchange'] + ' + ' + summary['Payment Method']

cheapest = summary.nsmallest(1, 'Total Cost (AUD)')[['Option', 'Total Cost (AUD)', 'Fee %']].values[0]
most_expensive = summary.nlargest(1, 'Total Cost (AUD)')[['Option', 'Total Cost (AUD)', 'Fee %']].values[0]
savings = most_expensive[1] - cheapest[1]

print(f'For 0.5 BTC purchase at ${btc_aud:,.0f}/BTC:\n')
print(f'✓ CHEAPEST: {cheapest[0]}')
print(f'  Cost: ${cheapest[1]:,.2f} | Fee: {cheapest[2]:.2f}%\n')
print(f'✗ MOST EXPENSIVE: {most_expensive[0]}')
print(f'  Cost: ${most_expensive[1]:,.2f} | Fee: {most_expensive[2]:.2f}%\n')
print(f'Potential Savings: ${savings:,.2f}')

---

## Reference

[https://protraderdaily.com](https://protraderdaily.com/crypto/how-to-buy-bitcoin-in-australia)
